# Generative Modeling: VAEs, GANs & Diffusion
**Level:** Professional | **Interactive Machine Learning Curriculum**

> Autoregressive models, Variational Autoencoders (ELBO, Reparameterization Trick), GANs (Minimax, WGAN), and Latent Diffusion (DDPM).

---

### 📖 Conceptual Foundations

#### Discriminative vs Generative Modeling

- **Discriminative Models:** Model the conditional probability $P(Y \mid X)$, learning a decision boundary to separate classes.
- **Generative Models:** Model the joint distribution $P(X, Y)$ or data distribution $P(X)$. They must understand the entire underlying structure of the data manifold to generate realistic new samples $x_{new} \sim P_{model}(X)$.


#### The Four Modern Generative Paradigms

| Paradigm | Core Formulation | Sampling Speed | Sample Quality / Mode Coverage | Dominant Applications |
| --- | --- | --- | --- | --- |
| Autoregressive Models | Factorizes joint distribution into product of conditionals: $P(X) = \prod_{t=1}^T P(x_t \mid x_{<t})$. | Slow (sequential $O(T)$ steps). | Exceptional density estimation; no mode collapse. | Large Language Models (GPT-4), Audio generation. |
| Variational Autoencoders (VAEs) | Maximizes Evidence Lower Bound (ELBO) via encoder $q_\phi(z|x)$ and decoder $p_\theta(x|z)$. | Instant ($1$ forward pass). | Smooth latent space; tends to produce blurry images due to MSE/Gaussian assumptions. | Latent space representation, compression, drug molecule design. |
| Generative Adversarial Networks (GANs) | Two-player minimax game between Generator $G(z)$ and Discriminator $D(x)$: $\min_G \max_D V(D, G)$. | Instant ($1$ forward pass). | Sharp, photo-realistic details; highly vulnerable to training instability and **Mode Collapse**. | Real-time face generation (StyleGAN), image editing. |
| Diffusion Models (DDPM / Score-Based) | Gradually corrupts data with Gaussian noise (Forward), then trains a network to invert the noise step-by-step (Reverse). | Historically slow (requires 20-50 denoising steps), but fast with modern ODE solvers. | State-of-the-art fidelity, zero mode collapse, stable training objective. | Image synthesis (Stable Diffusion, Midjourney, DALL-E 3), video generation (Sora). |


#### VAE: The Evidence Lower Bound (ELBO) & Reparameterization Trick

True marginal likelihood $\log p_\theta(x)$ is intractable. VAE maximizes the **ELBO**:

$$\log p_\theta(x) \ge \mathbb{E}_{q_\phi(z|x)}[\log p_\theta(x|z)] - D_{KL}(q_\phi(z|x) \parallel p(z))$$

- **Reconstruction Term:** $\mathbb{E}[\log p_\theta(x|z)]$ forces decoder to accurately reconstruct inputs.
- **Regularization Term:** $D_{KL}$ pulls the posterior $q_\phi(z|x)$ toward a standard Gaussian prior $\mathcal{N}(0, I)$, ensuring a continuous, interpolatable latent manifold.

> **The Reparameterization Trick:** Sampling $z \sim \mathcal{N}(\mu, \sigma^2)$ is stochastic and non-differentiable. By setting $z = \mu + \sigma \odot \epsilon$ where $\epsilon \sim \mathcal{N}(0, I)$, gradients flow smoothly into $\mu$ and $\sigma$ via standard backpropagation!


#### Latent Diffusion & Classifier-Free Guidance (CFG)

Running diffusion directly on high-resolution pixels ($1024 \times 1024$) is computationally ruinous. **Latent Diffusion Models (LDM / Stable Diffusion)** solve this by compressing images into a lower-dimensional latent space using a pretrained VAE, performing the iterative denoising diffusion process entirely in latent space.

**Classifier-Free Guidance (CFG, Ho & Salimans, 2022):** Balances sample quality and prompt adherence without an auxiliary classifier:

$$\tilde{\epsilon}_\theta(z_t, c) = \epsilon_\theta(z_t, \emptyset) + s \cdot (\epsilon_\theta(z_t, c) - \epsilon_\theta(z_t, \emptyset))$$

Where $s > 1$ (typically $7.0 - 9.0$) boosts conditioning signal $c$, creating sharp images aligned with the prompt.



In [ ]:
# %matplotlib inline
import numpy as np
import matplotlib.pyplot as plt

# Simulated VAE Encoder and Reparameterization Trick
def reparameterize(mu, logvar):
    std = np.exp(0.5 * logvar)
    eps = np.random.randn(*mu.shape) # Gaussian noise epsilon ~ N(0, I)
    return mu + std * eps

def kl_divergence(mu, logvar):
    # KL divergence between N(mu, var) and standard normal N(0, I)
    return -0.5 * np.sum(1 + logvar - np.square(mu) - np.exp(logvar), axis=-1)

# 4 samples with simulated mu and logvar in 2D latent space
np.random.seed(42)
mu = np.array([[2.0, 1.0], [-1.5, 0.5], [0.1, -2.0], [1.2, -1.0]])
logvar = np.array([[-0.2, 0.1], [0.0, -0.5], [0.2, 0.2], [-0.1, -0.1]])

z_samples = reparameterize(mu, logvar)
kl_losses = kl_divergence(mu, logvar)

print('Mean vector mu:\n', mu)
print('Sampled Latent Vectors z (differentiable):\n', np.round(z_samples, 3))
print('KL Divergence per sample:', np.round(kl_losses, 4))

# Visualize latent points
plt.figure(figsize=(6, 4))
plt.scatter(mu[:, 0], mu[:, 1], color='red', s=100, label='Mean Centers (mu)')
plt.scatter(z_samples[:, 0], z_samples[:, 1], color='blue', s=80, marker='x', label='Stochastic z Samples')
plt.title('VAE Reparameterization: Latent Embedding Manifold')
plt.xlabel('z_1'); plt.ylabel('z_2'); plt.grid(True); plt.legend()
plt.show()


---

### 🎯 Hands-On Experiments & Challenges
1. **Experiment with parameters:** Modify the hyperparameters or dataset sizes in the cell above.
2. **Add visual diagnostics:** Plot loss curves or feature distributions using Matplotlib/Seaborn.
3. **Test with new data:** Try running the model on synthetic noisy data or custom inputs.


In [ ]:
# Your custom experiment code here:

